# 🧠 Xiangqi-R1 Community GRPO Trainer — JRCP 3.0
## Đóng góp sức mạnh GPU cho mô hình cờ Tướng AI!

### 🔧 Hướng dẫn (3 bước):
1. **Gắn GPU T4**: Runtime → Change runtime type → T4 GPU
2. **Cài Secret**: 🔑 icon → `HF_TOKEN` = [token của bạn]
3. **Chạy tất cả**: Runtime → Run all

✅ Notebook sẽ tự động:
- Tải dataset JRCP 3.0 từ HuggingFace
- Huấn luyện Qwen 0.5B bằng GRPO (150 steps, ~30 phút)
- Push LoRA adapter lên HuggingFace Hub
- Hỗ trợ resume nếu bị ngắt

In [ ]:
# === XIANGQI-R1 COMMUNITY GRPO TRAINER — AUTO SETUP ===
import os, sys, signal, time, json, subprocess
from datetime import datetime

print("🧠 Xiangqi-R1 Community GRPO Trainer v3.0")
print("=" * 60)

# GPU Check
print("\n🔍 [1/3] Kiểm tra GPU...")
try:
    import torch
    if torch.cuda.is_available():
        gpu_name = torch.cuda.get_device_name(0)
        gpu_mem = torch.cuda.get_device_properties(0).total_mem / 1024**3
        print(f"   ✅ {gpu_name} ({gpu_mem:.1f} GB VRAM)")
    else:
        print("   ❌ GPU không khả dụng! Vui lòng gắn T4 GPU.")
        raise RuntimeError("Cần T4 GPU")
except ImportError:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'torch'])
    import torch

# HF Token
print("\n🔑 [2/3] Xác thực HuggingFace Token...")
HF_TOKEN = None
try:
    from google.colab import userdata
    HF_TOKEN = userdata.get('HF_TOKEN')
    if HF_TOKEN:
        print("   ✅ HF_TOKEN từ Colab Secrets")
except Exception:
    pass
if not HF_TOKEN:
    HF_TOKEN = os.environ.get('HF_TOKEN')
    if HF_TOKEN:
        print("   ✅ HF_TOKEN từ os.environ")
if not HF_TOKEN:
    HF_TOKEN = input("   Nhập HF_TOKEN: ").strip()
assert HF_TOKEN and len(HF_TOKEN) > 10, "❌ Cần HF_TOKEN!"
os.environ['HF_TOKEN'] = HF_TOKEN

# Install Unsloth + TRL
print("\n📦 [3/3] Cài đặt Unsloth + TRL (~2 phút)...")
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
    'unsloth', 'trl>=0.12.0', 'peft', 'accelerate', 'bitsandbytes',
    'datasets', 'huggingface_hub'], stdout=subprocess.DEVNULL)
print("   ✅ Unsloth + TRL installed")

print("\n" + "=" * 60)
print("✅ SETUP HOÀN TẤT!")

In [ ]:
# === NẠP DATASET JRCP 3.0 & MODEL ===
import os, json, re
from datasets import load_dataset, Dataset
from unsloth import FastLanguageModel

DATASET_REPO = "hoduyquocbao/xiangqi-r1-dataset"
MODEL_REPO = "hoduyquocbao/xiangqi-r1-0.5b"

# --- Nạp Dataset ---
print("📚 Nạp dataset JRCP 3.0...")
try:
    dataset = load_dataset(DATASET_REPO, split='train')
    print(f"   ✅ {len(dataset):,} mẫu từ {DATASET_REPO}")
except Exception as e:
    print(f"   ⚠️ Lỗi tải dataset: {e}")
    print("   🔄 Tạo dataset mẫu để demo...")
    sample = {
        "messages": [
            {"role": "system", "content": "Bạn là Xiangqi-R1 Master."},
            {"role": "user", "content": "FEN: rnbakabnr/9/1c5c1/p1p1p1p1p/9/9/P1P1P1P1P/1C5C1/9/RNBAKABNR w - - 0 1"},
            {"role": "assistant", "content": json.dumps({"thought": "<thought>Phân tích vị trí khai cuộc</thought>", "bestmove": "b2e2", "centipawn_eval": 50}, ensure_ascii=False)}
        ]
    }
    dataset = Dataset.from_list([sample] * 100)
    print(f"   ✅ Tạo {len(dataset)} mẫu demo")

# --- Nạp Model ---
print(f"\n🤖 Nạp model {MODEL_REPO}...")
try:
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=MODEL_REPO,
        max_seq_length=4096,
        dtype=None,
        load_in_4bit=True,
        token=os.environ['HF_TOKEN']
    )
    print(f"   ✅ Model loaded: {MODEL_REPO}")
except Exception:
    BASE_MODEL = "unsloth/Qwen2.5-Coder-0.5B-Instruct-bnb-4bit"
    print(f"   ⚠️ Fallback to {BASE_MODEL}")
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=BASE_MODEL,
        max_seq_length=4096,
        dtype=None,
        load_in_4bit=True
    )

# Cấu hình LoRA
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth"
)
print("✅ LoRA adapter cấu hình xong (r=16)")

In [ ]:
# === HUẤN LUYỆN GRPO 150 STEPS ===
import re, json, os
from trl import GRPOConfig, GRPOTrainer

# --- 3 Reward Functions ---

def reward_syntax(completions, **kwargs):
    """Chấm điểm cú pháp: JSON hợp lệ, có thought và bestmove."""
    scores = []
    for text in completions:
        score = 0.0
        content = text[0]["content"] if isinstance(text, list) else text
        try:
            obj = json.loads(content)
            score += 0.3  # Valid JSON
            if "thought" in obj and "<thought>" in str(obj["thought"]):
                score += 0.3  # Có thought tag
            if "bestmove" in obj:
                move = obj["bestmove"]
                if re.match(r'^[a-i][0-9][a-i][0-9]$', move):
                    score += 0.4  # Nước đi hợp lệ
        except (json.JSONDecodeError, TypeError, KeyError):
            pass
        scores.append(score)
    return scores

def reward_rule(prompts, completions, **kwargs):
    """Chấm điểm luật cờ: kiểm tra nước đi hợp lệ trên bàn cờ."""
    scores = []
    for prompt, text in zip(prompts, completions):
        score = 0.0
        content = text[0]["content"] if isinstance(text, list) else text
        try:
            obj = json.loads(content)
            move = obj.get("bestmove", "")
            # Kiểm tra định dạng nước đi
            if re.match(r'^[a-i][0-9][a-i][0-9]$', move):
                score += 0.5
                # Kiểm tra nước đi không trùng ô xuất phát
                if move[:2] != move[2:]:
                    score += 0.3
                # Kiểm tra ô trong bàn cờ 9x10
                from_col = ord(move[0]) - ord('a')
                from_row = int(move[1])
                to_col = ord(move[2]) - ord('a')
                to_row = int(move[3])
                if (0 <= from_col <= 8 and 0 <= from_row <= 9 and
                    0 <= to_col <= 8 and 0 <= to_row <= 9):
                    score += 0.2
        except (json.JSONDecodeError, TypeError, KeyError):
            pass
        scores.append(score)
    return scores

def reward_quality(prompts, completions, **kwargs):
    """Chấm điểm chất lượng suy luận: chiều sâu thought, candidates, comparison."""
    scores = []
    for text in completions:
        score = 0.0
        content = text[0]["content"] if isinstance(text, list) else text
        try:
            obj = json.loads(content)
            thought = str(obj.get("thought", ""))
            # Thưởng chiều dài thought (tối đa 0.3 cho >500 chars)
            thought_len = len(thought)
            score += min(0.3, thought_len / 1500.0)
            # Thưởng có candidates
            candidates = obj.get("candidates", [])
            if len(candidates) >= 2:
                score += 0.2
            if len(candidates) >= 3:
                score += 0.1
            # Thưởng có comparison
            if obj.get("comparison"):
                score += 0.2
            # Thưởng có centipawn_eval
            if "centipawn_eval" in obj:
                score += 0.2
        except (json.JSONDecodeError, TypeError, KeyError):
            pass
        scores.append(score)
    return scores

# --- Cấu hình GRPO ---
OUTPUT_DIR = "/content/grpo_output"
grpo_config = GRPOConfig(
    output_dir=OUTPUT_DIR,
    max_steps=150,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    learning_rate=5e-6,
    max_completion_length=2048,
    num_generations=4,
    logging_steps=5,
    save_steps=25,
    fp16=not torch.cuda.is_bf16_supported(),
    bf16=torch.cuda.is_bf16_supported(),
    optim="adamw_8bit",
    report_to="none",
    remove_unused_columns=False,
)

# --- Resume Support ---
resume_checkpoint = None
if os.path.exists(OUTPUT_DIR):
    checkpoints = sorted([d for d in os.listdir(OUTPUT_DIR) if d.startswith('checkpoint-')])
    if checkpoints:
        resume_checkpoint = os.path.join(OUTPUT_DIR, checkpoints[-1])
        print(f"🔄 Resume từ: {resume_checkpoint}")

# --- Huấn luyện ---
print("\n🏋️ Bắt đầu huấn luyện GRPO (150 steps, ~30 phút trên T4)...")
trainer = GRPOTrainer(
    model=model,
    processing_class=tokenizer,
    config=grpo_config,
    train_dataset=dataset,
    reward_funcs=[reward_syntax, reward_rule, reward_quality],
)

try:
    trainer.train(resume_from_checkpoint=resume_checkpoint)
    print("✅ Huấn luyện hoàn tất!")
except KeyboardInterrupt:
    print("\n⚠️ Bị ngắt! Checkpoint đã lưu, chạy lại để resume.")

# Lưu adapter
ADAPTER_DIR = "/content/community_adapter"
model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)
print(f"💾 Adapter lưu tại: {ADAPTER_DIR}")

In [ ]:
# === PUSH ADAPTER & CẢM ƠN ===
import os, time
from datetime import datetime
from huggingface_hub import HfApi

ADAPTER_DIR = "/content/community_adapter"
DATASET_REPO = "hoduyquocbao/xiangqi-r1-dataset"
adapter_file = os.path.join(ADAPTER_DIR, "adapter_model.safetensors")

if os.path.exists(adapter_file):
    api = HfApi(token=os.environ['HF_TOKEN'])
    timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
    remote_path = f"community/adapter_{timestamp}.safetensors"

    print(f"☁️ Đang push adapter lên {DATASET_REPO}...")
    api.upload_file(
        path_or_fileobj=adapter_file,
        path_in_repo=remote_path,
        repo_id=DATASET_REPO,
        repo_type="dataset",
        commit_message=f"[Community-Train] GRPO 150 steps | {timestamp}"
    )

    size_mb = os.path.getsize(adapter_file) / 1024 / 1024
    print(f"\n{'=' * 60}")
    print(f"  🏆 CẢM ƠN BẠN ĐÃ ĐÓNG GÓP GPU!")
    print(f"  🧠 Model: Qwen 0.5B + LoRA r=16")
    print(f"  🏋️ Training: GRPO 150 steps")
    print(f"  📁 Adapter: {size_mb:.1f} MB")
    print(f"  ☁️ Uploaded: {DATASET_REPO}/{remote_path}")
    print(f"{'=' * 60}")
    print(f"\n  Mỗi adapter bạn đóng góp giúp Xiangqi-R1 thông minh hơn!")
    print(f"  ❤️ Thank you for contributing GPU power!")
else:
    print("⚠️ Không tìm thấy adapter. Vui lòng chạy Cell 3 trước.")